# Quantum Generative AI with 2 Qubits
### A Variational Quantum Circuit that generates synthetic data
**Qiskit Fall Fest** · about 25 minutes · Qiskit 2.6

**Generative AI in one sentence:** learn what real data looks like (its *probability distribution*), then **sample** from what you learned to create new, realistic data.
ChatGPT does this with words. Today we do it with **2 qubits, 4 knobs and Qiskit**.

**Plan**
1. The real data we want to imitate
2. A quantum circuit is a random-number machine
3. The model: a parameterized quantum circuit
4. Reading the model's probabilities
5. The loss: how far are we from the data?
6. Training: the AI part
7. Generating synthetic data
8. Why entanglement matters
9. Your turn

In [ ]:
# Run once.
%pip install -q qiskit==2.6.1 pylatexenc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

import qiskit
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_distribution

print("Qiskit version:", qiskit.__version__)

## 1 · The real data we want to imitate

Imagine a survey of days in Kumasi during the rainy season. Each day we record two yes/no answers: **Was it raining? Did people carry an umbrella?**

Real data like this is often private, expensive or scarce. A generative model learns its pattern and can then produce as many **synthetic** records as we want, with the same statistics.

Two yes/no questions → **two qubits**: qubit 0 = rain, qubit 1 = umbrella.

| bitstring | meaning | share of days |
|:-:|---|:-:|
| `00` | no rain, no umbrella | 45% |
| `01` | rain, **no** umbrella (got wet!) | 10% |
| `10` | umbrella, no rain | 10% |
| `11` | rain **and** umbrella | 35% |

⚠️ Qiskit writes bitstrings **right to left**: qubit 0 is the right-most bit. So `01` means rain = 1, umbrella = 0.

In [ ]:
# Our "real data": the share of days in each situation
target = {"00": 0.45,   # no rain, no umbrella
          "01": 0.10,   # rain, no umbrella
          "10": 0.10,   # umbrella, no rain
          "11": 0.35}   # rain and umbrella

def describe(bits):
    # '01' -> 'rain, no umbrella'   (qubit 0 = right-most bit = rain)
    rain = "rain" if bits[1] == "1" else "no rain"
    umbrella = "umbrella" if bits[0] == "1" else "no umbrella"
    return f"{rain}, {umbrella}"

def plot(*dists, labels=None, title=""):
    # Bar chart of one or more distributions (counts or probabilities), using Qiskit's plot_distribution
    fig = plot_distribution(list(dists) if len(dists) > 1 else dists[0], legend=labels, title=title,
                            figsize=(8, 4), bar_labels=len(dists) == 1)
    fig.axes[0].set_ylabel("probability")
    return fig

plot(target, title="Real data: the distribution we want to learn")

Notice the pattern: on **80%** of days the two answers match (`00` or `11`). Rain and umbrellas go together. That relationship is a **correlation**, and our model will have to learn it.

## 2 · A quantum circuit is a random-number machine

Measuring 2 qubits gives one of 4 outcomes: `00`, `01`, `10`, `11`.
The **Hadamard gate (H)** puts a qubit in *superposition*: when measured, it is 0 or 1 with equal chance.
Qiskit's **Sampler** runs the circuit many times (*shots*). Each shot is one sample.

In [ ]:
# 1. a circuit with 2 qubits
# 2. put both qubits in superposition (H gate)
# 3. measure all qubits
# 4. sample it 1,000 times with StatevectorSampler and plot the result

**Measuring = sampling.** A quantum circuit *is* a probability distribution.
But 25% / 25% / 25% / 25% is not our data. We need **knobs** to reshape the probabilities.

## 3 · The model: a Parameterized Quantum Circuit (PQC)

**The knob: RY(θ)** rotates a qubit. The chance of measuring 1 is sin²(θ/2):

| angle θ | 0 | π/2 | π |
|---|:-:|:-:|:-:|
| chance of 1 | 0% | 50% | 100% |

Our generator has three parts:
- **RY on each qubit**: sets each qubit's own probability.
- **CNOT (CX)**: *entangles* the qubits. It links rain and umbrella.
- **RY on each qubit again**: fine-tunes the result.

The 4 angles θ[0]…θ[3] are the model's **trainable parameters**, like the weights of a neural network.
A PQC whose parameters are tuned by a classical optimizer is called a **Variational Quantum Circuit (VQC)**.

In [ ]:
# 1. four parameters: theta = ParameterVector("theta", 4)
# 2. RY on qubit 0 and on qubit 1
# 3. CNOT from qubit 0 to qubit 1
# 4. RY on qubit 0 and on qubit 1 again
# 5. draw it

## 4 · Reading the model's probabilities

To train the model we need its probability for each outcome. With 2 qubits we can compute them **exactly** using `Statevector`.
(This is a simulator shortcut. A real quantum computer would *estimate* them from many shots.)

In [ ]:
def model_probs(circuit, params):
    "Probabilities of 00, 01, 10, 11 for one setting of the knobs."
    p = Statevector(circuit.assign_parameters(params)).probabilities()
    return {format(i, "02b"): p[i] for i in range(4)}

In [ ]:
# 1. pick 4 random angles
# 2. get the model's probabilities with model_probs
# 3. plot them next to the real data with plot(...)

An untrained model produces the wrong distribution. Learning means **finding the 4 angles** that make the bars match.

## 5 · The loss: how far is the model from the data?

We measure the gap with the **KL divergence**:

$$\text{KL} = \sum_{x}\; p_{\text{data}}(x)\,\log\frac{p_{\text{data}}(x)}{p_{\text{model}}(x)}$$

- **0** means the model matches the data perfectly. Bigger means more different.
- It punishes the model for giving *low* probability to things that really happen.
- Training ChatGPT minimizes the same quantity (there it is called the *cross-entropy* or *negative log-likelihood*).

In [ ]:
def kl_divergence(data, model):
    return sum(p * np.log(p / (model[x] + 1e-12)) for x, p in data.items() if p > 0)

print(f"Loss of the untrained model: {kl_divergence(target, untrained):.2f}   (a perfect model scores 0)")

## 6 · Training: the AI part

Nobody tells the circuit which angles to use. It **learns them from the data** in a loop:

```
 ┌──────────────────┐   probabilities   ┌──────────────────────┐
 │  quantum circuit │ ────────────────▶ │  loss: compare with  │
 │  (4 angles)      │                   │  the real data       │
 └──────────────────┘                   └──────────────────────┘
          ▲            new angles                  │
          └───────── classical optimizer ◀─────────┘
```

This is a **hybrid quantum–classical** loop. The optimizer here is **COBYLA** from SciPy: it tries small changes to the angles and keeps the ones that lower the loss. The jumps at the start are the optimizer exploring; then it settles down.

In [ ]:
def train(circuit, data, seed=0):
    "Find the angles that make the circuit's distribution match the data."
    history = []

    def cost(params):
        loss = kl_divergence(data, model_probs(circuit, params))
        history.append(loss)
        return loss

    start = np.random.default_rng(seed).uniform(0, np.pi, circuit.num_parameters)
    result = minimize(cost, start, method="COBYLA")
    return result.x, history

In [ ]:
# 1. train the circuit on the target data
# 2. plot the loss history
# 3. print the learned angles

In [ ]:
# compare the trained model's probabilities with the real data

The bars match. The circuit has learned the distribution, **including the rain–umbrella correlation**.

## 7 · Generating synthetic data

Now we **use** the generator: add measurements and run it. Every shot is a brand-new synthetic record.

In [ ]:
# 1. add measurements to the trained circuit
# 2. sample it 1,000 times with the learned angles
# 3. print the first 8 synthetic records in words

In [ ]:
# compare the synthetic samples with the real data

The synthetic data follows the real pattern. Small differences are normal: sampling is random, just as 10 coin flips rarely give exactly 5 heads. More shots → closer.

## 8 · Why the entangling gate matters

Remove the CNOT and each qubit is on its own. The model can still say *"it rains on X% of days"* and *"people carry umbrellas on Y% of days"*, but it **cannot** say *"umbrellas come with rain"*.

In [ ]:
# 1. the same circuit WITHOUT the CNOT
# 2. train it on the same data
# 3. plot: real data vs with CNOT vs without CNOT

Without the CNOT the best the circuit can do is about 30% / 25% / 25% / 20%. It still gets the share of rainy days right (45%), but it has **lost the link** between rain and umbrellas.

**Entanglement is how a quantum circuit stores correlations.**

## 9 · Your turn

Change the data and retrain. Some ideas:
- `{"00": 0.5, "01": 0.0, "10": 0.0, "11": 0.5}`: always both or neither (a perfect correlation, the famous **Bell state**)
- `{"00": 0.0, "01": 0.5, "10": 0.5, "11": 0.0}`: always the opposite
- your own numbers (they must add up to 1)

In [ ]:
my_data = {"00": 0.5, "01": 0.0, "10": 0.0, "11": 0.5}

my_params, _ = train(pqc, my_data)
my_counts = sampler.run([(generator, my_params)], shots=1000).result()[0].data.meas.get_counts()
plot(my_data, my_counts, labels=["my data", "generated"], title="My generator")

## Recap

| Generative AI idea | In this notebook |
|---|---|
| Training data | The rain/umbrella distribution |
| Model with weights | A parameterized quantum circuit with 4 angles |
| Loss | KL divergence |
| Learning | COBYLA optimizer in a hybrid quantum–classical loop |
| Generating | Measuring the circuit (Sampler shots) |
| Learning relationships | Entanglement (the CNOT) |

**Honest note:** 2 qubits are easy for any laptop, so there is no quantum advantage here. With *n* qubits, a circuit holds a distribution over 2ⁿ outcomes (50 qubits → about 10¹⁵), and some quantum circuits produce distributions believed to be very hard to sample classically. Whether that leads to *better* generative AI is an open research question.

This kind of model is called a **Quantum Circuit Born Machine**. Further reading: [Benedetti et al., *A generative modeling approach for benchmarking and training shallow quantum circuits* (npj Quantum Information, 2019)](https://arxiv.org/abs/1801.07686).

---
## Bonus (after the session): a noisy model of a real IBM quantum computer

`FakeFez` is a snapshot of IBM's 156-qubit `ibm_fez` processor, including its measured noise. Running our trained generator on it shows what real hardware does to the distribution.

In [ ]:
%pip install -q qiskit-ibm-runtime

In [ ]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.executor_sampler import Sampler    # the current Runtime Sampler
from qiskit_ibm_runtime.fake_provider import FakeFez

backend = FakeFez()
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(generator.assign_parameters(trained_params))   # rewrite for the chip

noisy_counts = Sampler(mode=backend).run([isa_circuit], shots=1000).result()[0].data.meas.get_counts()
plot(target, noisy_counts, labels=["real data", "noisy IBM chip (simulated)"],
     title="Generated on a noisy quantum computer")

Noise blurs the distribution a little, but a 2-qubit circuit is short, so it survives well.

**On a real device:** create a free account at [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com), then replace `backend = FakeFez()` with:

```python
from qiskit_ibm_runtime import QiskitRuntimeService
# first time only: QiskitRuntimeService.save_account(token="API_KEY", instance="INSTANCE_CRN")
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
```